In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from dataclasses import dataclass, replace
from scipy.optimize import brentq
import warnings
warnings.filterwarnings('ignore')

# Visual style & palette constants
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor']   = '#333333'
plt.rcParams['axes.linewidth']   = 0.8
plt.rcParams['grid.color']       = '#cccccc'
plt.rcParams['grid.linestyle']   = '--'
plt.rcParams['grid.alpha']       = 0.5

C_IPR = '#1f77b4'   # deep blue
C_VLP = '#d62728'   # crimson
C_OP  = '#2ca02c'   # forest green
C_GAS = '#ff7f0e'   # amber
C_AOF = '#9467bd'   # purple
C_FB  = '#8c564b'   # brown
C_WC  = '#17becf'   # cyan/teal


In [ ]:
# Global configuration & API 5CT Tubing Catalogues
import numpy as np
import pandas as pd

WELL_CONFIG = dict(well_type='Vertical', angle_deg=90.0, survey=None)

# Standard API 5CT Tubing Catalogues (by well type)
TUBING_CATALOG = {
    'Vertical': [
        {'label': '1.900" (1.610 in ID)', 'OD_in': 1.900, 'ID_in': 1.610, 'weight_lbft': 2.75, 'default': False},
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': True},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': False},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
    ],
    'Horizontal': [
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': False},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': True},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
        {'label': '4-1/2" (4.000 in ID)', 'OD_in': 4.500, 'ID_in': 4.000, 'weight_lbft': 11.60, 'default': False},
    ],
    'Directional': [
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': True},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': False},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
    ],
    'Custom': [
        {'label': '2-3/8" (1.995 in ID)', 'OD_in': 2.375, 'ID_in': 1.995, 'weight_lbft': 4.60, 'default': False},
        {'label': '2-7/8" (2.441 in ID)', 'OD_in': 2.875, 'ID_in': 2.441, 'weight_lbft': 6.50, 'default': True},
        {'label': '3-1/2" (2.992 in ID)', 'OD_in': 3.500, 'ID_in': 2.992, 'weight_lbft': 9.30, 'default': False},
        {'label': '4-1/2" (3.958 in ID)', 'OD_in': 4.500, 'ID_in': 3.958, 'weight_lbft': 12.60, 'default': False},
    ],
}

def get_default_tubing(well_type='Vertical'):
    """Return the default tubing entry for a well type."""
    cat = TUBING_CATALOG.get(well_type, TUBING_CATALOG['Vertical'])
    for t in cat:
        if t.get('default'):
            return t
    return cat[0]

def get_tubing_sensitivity_list(well_type='Vertical'):
    """Return {label: ID_in} dict for all tubing sizes for a well type."""
    cat = TUBING_CATALOG.get(well_type, TUBING_CATALOG['Vertical'])
    return {t['label']: t['ID_in'] for t in cat}

def _min_curve(md, inc, azi):
    """Minimum curvature trajectory calculations for directional wells."""
    md = np.asarray(md, float); inc = np.asarray(inc, float); azi = np.asarray(azi, float)
    n = len(md); tvd = np.zeros(n); north = np.zeros(n); east = np.zeros(n)
    for i in range(1, n):
        dmd = md[i] - md[i-1]; a1, a2 = np.radians(inc[i-1]), np.radians(inc[i])
        f1, f2 = np.radians(azi[i-1]), np.radians(azi[i])
        dl = np.arccos(np.clip(np.cos(a2-a1) - np.sin(a1)*np.sin(a2)*(1-np.cos(f2-f1)), -1, 1))
        rf = (2/dl * np.tan(dl/2)) if dl > 1e-9 else 1.0
        tvd[i] = tvd[i-1] + dmd/2 * (np.cos(a1) + np.cos(a2)) * rf
        north[i] = north[i-1] + dmd/2 * (np.sin(a1)*np.cos(f1) + np.sin(a2)*np.cos(f2)) * rf
        east[i] = east[i-1] + dmd/2 * (np.sin(a1)*np.sin(f1) + np.sin(a2)*np.sin(f2)) * rf
    return tvd, north, east

def build_survey_df(md, inc, azi, unit='ft'):
    """Construct survey DataFrame with TVD and Cartesian offsets."""
    md = np.asarray(md, float); inc = np.asarray(inc, float); azi = np.asarray(azi, float)
    if unit.lower() in ('m', 'meter', 'metre', 'meters', 'metres'): md = md * 3.28084
    tvd, north, east = _min_curve(md, inc, azi)
    return pd.DataFrame({'MD_ft': md, 'TVD_ft': tvd, 'INC_survey_deg': inc,
                         'INC_code_deg': 90.0 - inc, 'AZI_deg': azi, 'North_ft': north, 'East_ft': east})
